In [1]:
import sys
from pathlib import Path
root = Path().resolve().parent  # adjust level as needed
sys.path.insert(0, str(root))

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from datasets import load_dataset

ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

In [7]:
ds_sample = ds["train"].shuffle(seed=42).select(range(20))

In [8]:
ds_sample[0]

{'id': '5ae3cfe05542990afbd1e1e3',
 'question': 'Which airport is located in Maine, Sacramento International Airport or Knox County Regional Airport?',
 'answer': 'Knox County Regional Airport',
 'type': 'comparison',
 'level': 'medium',
 'supporting_facts': {'title': ['Sacramento International Airport',
   'Knox County Regional Airport'],
  'sent_id': [0, 0]},
 'context': {'title': ['Vinalhaven, Maine',
   'Owls Head, Maine',
   'North Haven, Maine',
   'Downeast Flight 46',
   'Northern California TRACON',
   'Sacramento International Airport',
   'Knox County Regional Airport',
   'Matinicus Isle, Maine',
   'Raleigh Executive Jetport',
   'Lea County Regional Airport'],
  'sentences': [['Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States.',
    ' Vinalhaven is also used to refer to the Island itself.',
    ' The population was 1,165 at the 2010 census.',
    ' It is home to a thriving lobster fishery and hosts a summer colony.',
 

In [9]:
from langchain_core.documents import Document

In [10]:
all_documents = []

for doc_idx, sample in enumerate(ds_sample):
    raws = sample['context']['sentences']
    # Attach elements in same array (sentences), add linebreak between arrays (paragraphs)
    full_doc_content = "\n\n".join(["".join(paragraph) for paragraph in raws])
    
    for chunk_idx, paragraph_sentences in enumerate(raws):
        # Attach elements in same array (sentences)
        chunk_content = "".join(paragraph_sentences)
        
        chunk_id = f"doc_{doc_idx + 1}_chunk_{chunk_idx + 1}"
        doc = Document(
            id=chunk_id, 
            page_content=chunk_content,
            metadata={
                "doc_idx": doc_idx + 1,
                "chunk_id": f"doc_{doc_idx + 1}_chunk_{chunk_idx + 1}",
                "doc_content": full_doc_content,
                "title": sample['context']['title'][chunk_idx],
                "question": sample['question'],
                "answer": sample['answer']
            }
        )
        all_documents.append(doc)

print(f"Created {len(all_documents)} documents from {len(ds_sample)} samples.")

Created 198 documents from 20 samples.


In [11]:
# Inspect the first document
all_documents[0].metadata['doc_content']

'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The population was 355 at the 2010 census. North Haven is accessed by three-times daily ferry service from Rockland, or by air

In [12]:
len(all_documents)

198

In [13]:
import pickle

In [14]:
with open(root / "datasets" / "hotpotqa.pkl", "wb") as f:
    pickle.dump(all_documents, f)

In [15]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-mini", temperature=0.0)

In [16]:
all_documents[0]

Document(id='doc_1_chunk_1', metadata={'doc_idx': 1, 'chunk_id': 'doc_1_chunk_1', 'doc_content': 'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The population was 355 at the

In [17]:
DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""
 
CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>
 
Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""
 
 
def situate_context(chunks: list[Document]) -> dict[str, str]:
    prompts = []
    for chunk in chunks:
        prompt = [
            {"role": "system", "content": "You MUST answer in Korean."},
                {"role": "user", "content": DOCUMENT_CONTEXT_PROMPT.format(doc_content=chunk.metadata["doc_content"])},
                {"role": "user", "content": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk.page_content)},
            ]
        prompts.append(prompt)
    response = llm.batch(prompts)
    return response

In [18]:
res = situate_context(all_documents)

In [19]:
for r, chunk in zip(res, all_documents):
    chunk.page_content =  r.content + "\n\n" + chunk.page_content
    chunk.metadata["contextualized_content"] = r.content + "\n\n" + chunk.page_content
    chunk.metadata["original_content"] = chunk.page_content   

In [20]:
all_documents[0]

Document(id='doc_1_chunk_1', metadata={'doc_idx': 1, 'chunk_id': 'doc_1_chunk_1', 'doc_content': 'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The population was 355 at the

In [21]:
import uuid

In [22]:
for doc in all_documents:
    doc.id = str(uuid.uuid4())

In [23]:
all_documents[:5]

[Document(id='abbae568-5779-4235-9811-89af16472aa9', metadata={'doc_idx': 1, 'chunk_id': 'doc_1_chunk_1', 'doc_content': 'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The p

In [24]:
with open(root / "datasets" / "hotpotqa.pkl", "wb") as f:
    pickle.dump(all_documents, f)

In [25]:
# Check the structure of supporting_facts
ds_sample[0]['supporting_facts']

{'title': ['Sacramento International Airport', 'Knox County Regional Airport'],
 'sent_id': [0, 0]}

In [26]:
# Check the context titles for comparison
ds_sample[0]['context']['title']

['Vinalhaven, Maine',
 'Owls Head, Maine',
 'North Haven, Maine',
 'Downeast Flight 46',
 'Northern California TRACON',
 'Sacramento International Airport',
 'Knox County Regional Airport',
 'Matinicus Isle, Maine',
 'Raleigh Executive Jetport',
 'Lea County Regional Airport']

In [27]:
# Create evaluation dataset
# Golden chunks are identified by matching supporting_facts titles with context titles

eval_dataset = []

for doc_idx, sample in enumerate(ds_sample):
    # Get the titles of supporting facts (golden paragraphs)
    golden_titles = set(sample['supporting_facts']['title'])
    context_titles = sample['context']['title']
    
    # Find chunk indices where title matches a supporting fact title
    golden_chunk_ids = []
    for chunk_idx, title in enumerate(context_titles):
        if title in golden_titles:
            chunk_id = f"doc_{doc_idx + 1}_chunk_{chunk_idx + 1}"
            golden_chunk_ids.append(chunk_id)
    
    eval_entry = {
        "query": sample['question'],
        "answer": sample['answer'],
        "golden_chunk_ids": golden_chunk_ids,
        "doc_idx": doc_idx + 1
    }
    eval_dataset.append(eval_entry)

print(f"Created eval dataset with {len(eval_dataset)} entries.")

Created eval dataset with 20 entries.


In [28]:
# Inspect the first few eval entries
eval_dataset[:3]

[{'query': 'Which airport is located in Maine, Sacramento International Airport or Knox County Regional Airport?',
  'answer': 'Knox County Regional Airport',
  'golden_chunk_ids': ['doc_1_chunk_6', 'doc_1_chunk_7'],
  'doc_idx': 1},
 {'query': 'Peter Hobbs founded the company that is based in what town in Manchester?',
  'answer': 'Failsworth',
  'golden_chunk_ids': ['doc_2_chunk_2', 'doc_2_chunk_3'],
  'doc_idx': 2},
 {'query': 'What direction does the river that Austrolebias bellotti are found in flow?',
  'answer': 'north to south',
  'golden_chunk_ids': ['doc_3_chunk_4', 'doc_3_chunk_5'],
  'doc_idx': 3}]

In [29]:
# Save eval dataset
import json

with open(root / "datasets" / "hotpotqa_eval.json", "w") as f:
    json.dump(eval_dataset, f, indent=2)
    
print("Saved eval dataset to datasets/hotpotqa_eval.json")

Saved eval dataset to datasets/hotpotqa_eval.json


In [30]:
import pickle
with open(root / "datasets" / "hotpotqa.pkl", "rb") as f:
    all_documents = pickle.load(f)
 

In [32]:
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="embedding-passage")
vectorstore = FAISS.from_documents(documents=all_documents, embedding=embeddings)
vectorstore.save_local(root / "faiss_index", "hotpotqa")

In [ ]:
import pandas as pd

def recall(df: pd.DataFrame, retrieved_docs: list[list]) -> dict:
    """
    Calculate recall for retrieval evaluation.
    A query is considered a true positive if ANY of the golden_chunk_ids are retrieved.
    """
    true_positives = 0
    false_negatives = 0

    for i, row in df.iterrows():
        golden_chunk_ids = row["golden_chunk_ids"]  # List of golden chunk IDs
        retrieved_chunk_ids = [doc.metadata["chunk_id"] for doc in retrieved_docs[i]]
        
        # Check if ANY golden chunk is in retrieved chunks
        if any(golden_id in retrieved_chunk_ids for golden_id in golden_chunk_ids):
            true_positives += 1
        else:
            print("index: ", i)
            print(row["query"])
            print("golden:", golden_chunk_ids)
            print("retrieved:", retrieved_chunk_ids)
            false_negatives += 1

    print(f"True Positives: {true_positives}, False Negatives: {false_negatives}")

    recall_score = true_positives / (true_positives + false_negatives)
    return {"recall": recall_score}

In [55]:
from langchain_upstage import UpstageEmbeddings
from reranker.rrf import ReciprocalRankFusion
from langchain_community.retrievers import BM25Retriever
embeddings = UpstageEmbeddings(model="embedding-passage")
bm25_retriever = BM25Retriever.from_documents(all_documents)
bm25_retriever.k = 20
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
        root / "faiss_index", 
        embeddings,
        "hotpotqa",
        allow_dangerous_deserialization=True  # needed in newer versions
    )
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})
def retrieve_document(question: str) -> list[str]:
    retrieved_docs_faiss = faiss_retriever.invoke(question)
    retrieved_docs_bm25 = bm25_retriever.invoke(question)
    retrieved_docs_faiss = ReciprocalRankFusion.calculate_rank_score(retrieved_docs_faiss)
    retrieved_docs_bm25 = ReciprocalRankFusion.calculate_rank_score(retrieved_docs_bm25)
    retrieved_docs = retrieved_docs_faiss + retrieved_docs_bm25
    rrf_docs = ReciprocalRankFusion.get_rrf_docs(retrieved_docs, cutoff=5)
    return rrf_docs

In [56]:
retrieved_docs = [retrieve_document(eval_dataset[i]["query"]) for i in range(len(eval_dataset))]

In [34]:
eval_dataset[0]

{'query': 'Which airport is located in Maine, Sacramento International Airport or Knox County Regional Airport?',
 'answer': 'Knox County Regional Airport',
 'golden_chunk_ids': ['doc_1_chunk_6', 'doc_1_chunk_7'],
 'doc_idx': 1}

In [57]:
def recall(eval_dataset: list[dict], retrieved_docs: list[str]) -> dict:
    true_positives = 0
    false_negatives = 0

    for i, data in enumerate(eval_dataset):  
        # 중복 페이지 제거
        # reference_page_number = list({int(page) for page in row["page_number"].strip("[]").split(",")})
        reference_chunk_id = data["golden_chunk_ids"]
        retrieved_chunk_id = [doc.metadata["chunk_id"] for doc in retrieved_docs[i]]

        for chunk_id in reference_chunk_id:
            if chunk_id in retrieved_chunk_id:
                true_positives += 1
            else:
                print("index: ", i)
                print(data["query"])
                print(data["golden_chunk_ids"])
                print(retrieved_chunk_id)
                false_negatives += 1

    print(f"True Positives: {true_positives}, False Negatives: {false_negatives}")

    recall = true_positives / (true_positives + false_negatives)
    return {"recall": recall}

In [58]:
recall(eval_dataset, retrieved_docs)

index:  0
Which airport is located in Maine, Sacramento International Airport or Knox County Regional Airport?
['doc_1_chunk_6', 'doc_1_chunk_7']
['doc_1_chunk_7', 'doc_1_chunk_1', 'doc_1_chunk_2', 'doc_1_chunk_3', 'doc_1_chunk_8']
index:  2
What direction does the river that Austrolebias bellotti are found in flow?
['doc_3_chunk_4', 'doc_3_chunk_5']
['doc_3_chunk_4', 'doc_3_chunk_9', 'doc_3_chunk_1', 'doc_3_chunk_6', 'doc_3_chunk_7']
index:  7
Sir Thomas Kyriell was executed after which battle from the Wars of the Roses?
['doc_8_chunk_6', 'doc_8_chunk_9']
['doc_8_chunk_6', 'doc_8_chunk_1', 'doc_8_chunk_2', 'doc_8_chunk_4', 'doc_8_chunk_7']
index:  8
Where is the fruit, part of the flowering plant species in the palm family Arecaceae, most popular?
['doc_9_chunk_3', 'doc_9_chunk_9']
['doc_9_chunk_3', 'doc_9_chunk_6', 'doc_9_chunk_1', 'doc_9_chunk_10', 'doc_9_chunk_4']
index:  10
An actress nominated for 20 academy awards was featured on a TV series about genealogical research of 12 ame

{'recall': 0.8}

In [41]:
eval_dataset[16]

{'query': 'What is the current name of the club that Agustin Lionel Allione plays for?',
 'answer': 'Esporte Clube Bahia',
 'golden_chunk_ids': ['doc_17_chunk_8', 'doc_17_chunk_10'],
 'doc_idx': 17}

In [51]:
all_documents[0].metadata
  

{'doc_idx': 1,
 'chunk_id': 'doc_1_chunk_1',
 'doc_content': 'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The population was 355 at the 2010 census. North Haven is accesse